In [10]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *
from textblob import TextBlob

#pd.set_option("display.max_colwidth', -1)
from time import time
import re
import string
import os

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

from nltk.sentiment.vader import SentimentIntensityAnalyzer
import unicodedata

In [11]:
df_lead_paragraph = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/DateSentiment.pkl", compression=None)

In [ ]:
df = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/TimeSentiment.pkl")

In [ ]:
df.sort_values(by="Date")

In [ ]:
df_lead_paragraph

In [ ]:
# %run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/Initialization/Sentiment/Functions.ipynb"

In [ ]:
# New words and values
new_words = {
    'crushes': 100,
    'beats': 5,
    'misses': -5,
    'trouble': -10,
    'falls': -100,
    "dividends":100,
    "dividend":100,
    "earning":100,
    "ipo":100,
    "alliance":100,
    "enhance":100,
    "enhanc":100
}
# Instantiate the sentiment intensity analyzer with the existing lexicon
vader = SentimentIntensityAnalyzer()
# Update the lexicon
vader.lexicon.update(new_words)

In [ ]:
pos_word_list=[]
neu_word_list=[]
neg_word_list=[]
df  = pd.DataFrame()

# Adding new columns to the data frame
df[['compound']] = ''
df[['neg']] = ''
df[['neu']] = ''
df[['pos']] = ''
df[['Sentiment']] = ''
df[["lead_paragraph"]] = ''
df[["polarity"]] = ''
df[["subjectivity"]] = ''
df[["polarityB"]] = ''
df[['GI_Negative']] = ''
df[['GI_Positive']] = ''
df[['GI_Sentiment']] = ''
df[["GI_Polarity"]] = ''
df[["GI_Subjectivity"]] = ''

In [ ]:
# Reading the saved data pickle file
df_lead_paragraph = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/PressReleaseDateHourDF.pkl', compression=None)
# del df 

# 1) TRADING HOURS 09-16

In [ ]:
# Reading the saved data pickle file
df_lead_paragraph = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/PressReleaseDate_09_15.pkl', compression=None)

In [ ]:
df_lead_paragraph

In [ ]:
df  = pd.DataFrame()

# Adding new columns to the data frame
df[['compound']] = ''
df[['neg']] = ''
df[['neu']] = ''
df[['pos']] = ''
df[['sentiment']] = ''

In [ ]:
debug=False
countNeg=0
countNeu=0
countPos=0
countNegWord=0
countNeuWord=0
countPosWord=0

for date, row in df_lead_paragraph.T.iteritems():
    # print("The date is:"+str(date))
    # try:
    print(type(df_lead_paragraph.loc[date, 'lead_paragraph']))
    sentence = unicodedata.normalize('NFKD', row['lead_paragraph'])#.encode('ascii','ignore').decode('ascii')
    sentence = str(preprocessing_text(sentence))
    sentence = sentence.strip("[]")
    sentence = sentence.strip("''")
    df.at[date, 'lead_paragraph'] = sentence
    prc = row['NewsCount']

    ss = vader.polarity_scores(sentence)

    df.at[date, 'compound'] = ss['compound']
    df.at[date, 'neg'] = ss['neg']
    df.at[date, 'neu'] = ss['neu']
    df.at[date, 'pos'] = ss['pos']
    if ((ss['pos'] + ss['neg']) != 0):
        number = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
        if (number != ''):
            df.at[date, 'sentiment'] = ((100*(ss['neg'] / (ss['pos'] + ss['neg'])))*(prc/mean_prc))

    countNeg+=ss['neg']
    countNeu+=ss['neu']
    countPos+=ss['pos']

In [ ]:
df_lead_paragraph.drop(columns={"lead_paragraph"},inplace=True)
df = df.join(df_lead_paragraph)

In [ ]:
print("Number of Negative words="+str(countNeg))
print("Number of Positive words="+str(countPos))
print("Number of Neutral words="+str(countNeu))

df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Sentiment_09_15.pkl")

In [ ]:
del df

# 2) Not Trading Hours 16-09

In [ ]:
df_lead_paragraph = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/PressReleaseDate_16_08.pkl', compression=None)

In [ ]:
df_lead_paragraph

In [ ]:
df  = pd.DataFrame()

# Adding new columns to the data frame
df[['compound']] = ''
df[['neg']] = ''
df[['neu']] = ''
df[['pos']] = ''
df[['sentiment']] = ''

In [ ]:
debug=False
countNeg=0
countNeu=0
countPos=0
countNegWord=0
countNeuWord=0
countPosWord=0

for date, row in df_lead_paragraph.T.iteritems():
    # print("The date is:"+str(date))
    # try:
    print(type(df_lead_paragraph.loc[date, 'lead_paragraph']))
    sentence = unicodedata.normalize('NFKD', df_lead_paragraph.loc[date, 'lead_paragraph'])#.encode('ascii','ignore').decode('ascii')
    sentence = str(preprocessing_text(sentence))
    sentence = sentence.strip("[]")
    sentence = sentence.strip("''")
    df.at[date, 'lead_paragraph'] = sentence
    prc = row['NewsCount']

    ss = vader.polarity_scores(sentence)

    #print(sentence)
    #print("#2")
    df.at[date, 'compound'] = ss['compound']
    df.at[date, 'neg'] = ss['neg']
    df.at[date, 'neu'] = ss['neu']
    df.at[date, 'pos'] = ss['pos']
    if ((ss['pos'] + ss['neg']) != 0):
        number = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
        if (number != ''):
            df.at[date, 'sentiment'] = ((100*(ss['neg'] / (ss['pos'] + ss['neg'])))*(prc/mean_prc))

    countNeg+=ss['neg']
    countNeu+=ss['neu']
    countPos+=ss['pos']

In [ ]:
df_lead_paragraph.drop(columns={"lead_paragraph"},inplace=True)
df = df.join(df_lead_paragraph)

In [ ]:
print("Number of Negative words="+str(countNeg))
print("Number of Positive words="+str(countPos))
print("Number of Neutral words="+str(countNeu))

df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Sentiment_16_08.pkl")

In [ ]:
del df